# ML-04 — Search Intelligence Data Contract

## Note
The original assignment uses the FlyRank Hugging Face warehouse dataset. Due to repeated access issues with the gated dataset, I completed this notebook using the dataset provided inside the internship repository:

`../../data/raw/content_refresh_anonymized.csv`

This CSV contains the same type of anonymized search performance data required for learning the concepts of data contracts, feature engineering, and data leakage. All queries, feature engineering, and verification steps in this notebook are therefore performed on the repository CSV instead of the Hugging Face warehouse tables.


## 1. Unit of analysis + time window

### Unit of analysis

One row represents the search performance of one content page (URL) for a reporting record in the repository dataset.

### Time window

This notebook uses the complete time period available in the repository dataset (`content_refresh_anonymized.csv`) because the Hugging Face warehouse dataset was inaccessible.

### Prediction objective

Predict whether a page is likely to receive high traffic based on historical search performance.

### Deliberately excluded

Any future information (future clicks, impressions, or rankings) is excluded because it would introduce data leakage.

In [2]:
import pandas as pd

# Load repository dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst Five Rows")
display(df.head())

print("\nVerification")

print("Total Rows:", len(df))


Dataset Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First Five Rows


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Verification
Total Rows: 30000


## 2. Fields: feature / label / context / excluded

### Features

- clicks
- impressions
- ctr
- position
- search_volume

### Label

High traffic page (created from clicks above the dataset median).

### Context

- url
- client_id
- locale
- keyword_locale

These fields identify the content but are not used directly as prediction features.

### Excluded

Future information is excluded because it would leak the answer into the model.

In [5]:
# Create binary label

median_clicks = df["clicks_90d"].median()

df["label"] = (df["clicks_90d"] > median_clicks).astype(int)

print("Median Clicks:", median_clicks)

display(
    df[
        [
            "clicks_90d",
            "impressions_90d",
            "ctr",
            "avg_position",
            "search_volume",
            "label",
        ]
    ].head()
)

Median Clicks: 1.0


,clicks_90d,impressions_90d,ctr,avg_position,search_volume,label
0,29,3803,0.76,10.6,10.0,1
1,7,15320,0.05,20.3,90.0,1
2,11,12581,0.09,36.5,0.0,1
3,58,11751,0.49,6.2,10.0,1
4,24,19140,0.13,44.0,0.0,1


## 3. Verify it with queries (grain, counts, missing values, windows)

The following queries verify:

1. Dataset grain (one row per content performance record)
2. Number of rows and unique URLs
3. Missing values in selected features

In [7]:
# Query 01
print("Total Rows:", len(df))
print("Unique Content IDs:", df["content_id"].nunique())

# Query 02
display(
    df[
        [
            "clicks_90d",
            "impressions_90d",
            "search_volume",
            "avg_position",
            "ctr"
        ]
    ].describe()
)

# Query 03
available = df[df["impressions_90d"] > 0]

print("Rows with available impressions:", len(available))

Total Rows: 30000
Unique Content IDs: 30000


,clicks_90d,impressions_90d,search_volume,avg_position,ctr
count,30000.000000,30000.000000,27532.000000,30000.00000,30000.000000
mean,16.097333,5200.366300,158.882391,16.34238,0.510733
std,75.076958,16838.019547,1518.270825,15.21679,3.279162
min,0.000000,1.000000,0.000000,0.00000,0.000000
25%,0.000000,81.000000,0.000000,6.20000,0.000000
50%,1.000000,731.000000,10.000000,10.80000,0.070000
75%,7.000000,3615.250000,20.000000,22.30000,0.290000
max,4178.000000,517715.000000,74000.000000,245.00000,100.000000


Rows with available impressions: 30000


## 4. Data limits

### Limitation

This notebook uses the repository CSV instead of the complete warehouse dataset because the Hugging Face warehouse could not be accessed.

The repository dataset contains only the available exported search performance information, so additional warehouse features cannot be analyzed.

In [8]:
print("Dataset Limitations")

print("- Repository CSV used instead of warehouse tables.")
print("- Analysis is limited to available columns.")
print("- Future information was intentionally excluded to avoid leakage.")

Dataset Limitations
- Repository CSV used instead of warehouse tables.
- Analysis is limited to available columns.
- Future information was intentionally excluded to avoid leakage.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.